# 商品类目机会分析与用户分群联动策略

本 Notebook 基于淘宝 `UserBehavior.csv` 全量行为数据和第五部分生成的 `user_segmentation.csv`，使用 `chunksize` 分块读取方式完成商品类目机会分析，并结合用户分群结果输出增长策略。

注意：`category_id` 只是数字编号，没有真实类目名称。本分析只使用类目编号，不编造真实类目名称。

## 一、分析目标

从电商平台增长分析视角，本 Notebook 重点回答：

1. 哪些类目流量最高？
2. 哪些类目购买量最高？
3. 哪些类目浏览量高但购买转化低？
4. 哪些类目加购多但购买少？
5. 哪些类目收藏多但购买少？
6. 哪些类目转化率较高，适合重点推荐？
7. 不同用户分群分别主要浏览、加购、购买哪些类目？
8. 针对不同“类目机会 × 用户群体”，应该提出什么运营策略？

## 二、导入库

导入数据处理、可视化、字典和计数器工具。

In [ ]:
# 导入数据处理工具包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import defaultdict, Counter

# 导入 Markdown 展示工具，用于输出动态业务解读
from IPython.display import Markdown, display

# 设置中文字体，避免中文图表乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 设置 seaborn 图表风格
sns.set_theme(style="whitegrid", font="SimHei")

## 三、设置路径和调试参数

默认开启调试模式，只处理前 3 个 chunks，便于先验证代码能否跑通。确认无误后，可将 `DEBUG=False`、`MAX_CHUNKS=None` 后再跑全量数据。

In [ ]:
# 原始行为数据路径
raw_path = r"E:\ecommerce-user-growth-analysis\data\raw\UserBehavior.csv"

# 第五部分用户分群结果路径
user_seg_path = r"E:\ecommerce-user-growth-analysis\data\processed\user_segmentation.csv"

# 类目指标输出路径
category_metrics_path = r"E:\ecommerce-user-growth-analysis\data\processed\category_metrics.csv"

# 类目机会长表输出路径
category_opportunity_path = r"E:\ecommerce-user-growth-analysis\data\processed\category_opportunity.csv"

# 四类机会榜单输出路径
high_pv_low_buy_path = r"E:\ecommerce-user-growth-analysis\data\processed\high_pv_low_buy_categories.csv"
high_cart_low_buy_path = r"E:\ecommerce-user-growth-analysis\data\processed\high_cart_low_buy_categories.csv"
high_fav_low_buy_path = r"E:\ecommerce-user-growth-analysis\data\processed\high_fav_low_buy_categories.csv"
high_conversion_path = r"E:\ecommerce-user-growth-analysis\data\processed\high_conversion_categories.csv"

# 类目 × 用户分群行为输出路径
category_segment_behavior_path = r"E:\ecommerce-user-growth-analysis\data\processed\category_segment_behavior.csv"

# 类目策略建议输出路径
strategy_path = r"E:\ecommerce-user-growth-analysis\data\processed\category_strategy_recommendations.csv"

# 原始数据没有表头，因此手动指定列名
col_names = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]

# 分块读取大小：每次读取 100 万行，避免一次性读取 3.67GB 数据
chunksize = 1000000

# 调试参数：默认只跑前 3 个 chunk，确认无误后改为 DEBUG=False、MAX_CHUNKS=None
DEBUG = True
MAX_CHUNKS = 3

# 确保输出目录存在
processed_dir = os.path.dirname(category_metrics_path)
os.makedirs(processed_dir, exist_ok=True)

# 行为类型顺序，后续表格和图表都按这个顺序展开
behavior_order = ["pv", "fav", "cart", "buy"]

# 多榜单机会类型展示顺序
opportunity_order = [
    "高浏览低购买类目",
    "高加购低购买类目",
    "高收藏低购买类目",
    "高转化优势类目",
]

print("原始数据路径：", raw_path)
print("用户分群结果路径：", user_seg_path)
print("调试模式：", DEBUG, "最大处理 chunk 数：", MAX_CHUNKS)


## 四、读取用户分群结果

读取第五部分生成的 `user_segmentation.csv`，只保留 `user_id` 和 `user_segment` 两列，以降低内存占用。若缺少 `user_segment` 字段，需要先完成第五部分用户分群分析。

In [ ]:
# 检查用户分群结果是否存在
if not os.path.exists(user_seg_path):
    raise FileNotFoundError("找不到 user_segmentation.csv，请先运行第五部分用户分群分析。")

# 先读取少量数据检查字段名，避免字段名不一致导致后续报错
user_seg_columns = pd.read_csv(user_seg_path, nrows=0).columns.tolist()
print("user_segmentation.csv 字段：", user_seg_columns)

# 如果缺少必要字段，直接提示需要先完成第五部分
if "user_id" not in user_seg_columns or "user_segment" not in user_seg_columns:
    raise ValueError("user_segmentation.csv 中缺少 user_id 或 user_segment 字段，请先完成第五部分用户分群分析，并检查字段名。")

# 只读取必要字段，降低内存占用
user_segmentation = pd.read_csv(user_seg_path, usecols=["user_id", "user_segment"])

# 对 user_id 去重，避免 merge 后重复放大行为数量
user_segmentation = user_segmentation.drop_duplicates(subset=["user_id"])

# 展示用户分群样例
display(user_segmentation.head())
print(f"用户分群记录数：{len(user_segmentation):,}")

## 五、分块读取原始行为数据，统计类目表现

这一部分使用 `chunksize` 分块读取全量 `UserBehavior.csv`。每个分块处理完成后，只把累计结果保存在字典和集合中，不把全量原始数据放入内存。

同时完成两类统计：

1. 类目整体表现：各类目的浏览、收藏、加购、购买、用户数、独立商品数。
2. 类目 × 用户分群行为：不同用户群体在不同类目上的行为贡献。

In [ ]:
# 检查原始数据是否存在
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"找不到原始数据文件：{raw_path}")

# 每个类目的总行为数
category_total_counter = Counter()

# 每个类目的四类行为数量，例如 category_behavior_counter[category_id]['pv']
category_behavior_counter = defaultdict(Counter)

# 每个类目下，不同行为对应的去重用户集合
category_behavior_user_sets = defaultdict(lambda: defaultdict(set))

# 每个类目下被互动过的独立商品集合
category_item_sets = defaultdict(set)

# 类目 × 用户分群 × 行为类型的行为数量
category_segment_behavior_counter = defaultdict(Counter)

# 记录处理进度
total_rows = 0
matched_rows = 0

# 使用 chunksize 分块读取原始行为数据
reader = pd.read_csv(
    raw_path,
    names=col_names,
    header=None,
    chunksize=chunksize,
)

# 逐个 chunk 处理
for chunk_no, chunk in enumerate(reader, start=1):
    # 调试模式下只处理前 MAX_CHUNKS 个 chunk
    if DEBUG and MAX_CHUNKS is not None and chunk_no > MAX_CHUNKS:
        print(f"DEBUG=True，已处理前 {MAX_CHUNKS} 个 chunk，提前停止。")
        break

    # 累计原始处理行数
    total_rows += len(chunk)

    # 类目总行为数
    category_total_counter.update(chunk.groupby("category_id").size().to_dict())

    # 类目下四类行为数量
    category_behavior_counts = chunk.groupby(["category_id", "behavior_type"]).size()
    for (category_id, behavior_type), count in category_behavior_counts.items():
        category_behavior_counter[category_id][behavior_type] += int(count)

    # 类目下各行为的去重用户数：先对 category_id、behavior_type、user_id 去重，再放入集合
    category_behavior_user_pairs = chunk[["category_id", "behavior_type", "user_id"]].drop_duplicates()
    for (category_id, behavior_type), users in category_behavior_user_pairs.groupby(["category_id", "behavior_type"])["user_id"]:
        category_behavior_user_sets[category_id][behavior_type].update(users.tolist())

    # 类目下互动过的独立商品数：先对 category_id、item_id 去重，再放入集合
    category_item_pairs = chunk[["category_id", "item_id"]].drop_duplicates()
    for category_id, item_ids in category_item_pairs.groupby("category_id")["item_id"]:
        category_item_sets[category_id].update(item_ids.tolist())

    # 关联用户分群，用于分析不同用户群体在类目上的行为贡献
    chunk_with_segment = chunk.merge(user_segmentation, on="user_id", how="inner")
    matched_rows += len(chunk_with_segment)

    # 统计 category_id、user_segment、behavior_type 维度的行为数量
    segment_behavior_counts = chunk_with_segment.groupby(["category_id", "user_segment", "behavior_type"]).size()
    for (category_id, user_segment, behavior_type), count in segment_behavior_counts.items():
        category_segment_behavior_counter[(category_id, user_segment)][behavior_type] += int(count)

    # 打印进度，便于观察长任务运行情况
    print(f"已处理第 {chunk_no} 个 chunk，累计处理 {total_rows:,} 行，匹配用户分群 {matched_rows:,} 行")

print("\n分块读取完成")
print(f"累计处理行数：{total_rows:,}")
print(f"累计匹配用户分群行数：{matched_rows:,}")
print(f"累计类目数：{len(category_total_counter):,}")

## 六、计算类目转化指标

将累计统计结果整理为 `category_metrics` 表，并计算行为转化率和用户转化率。所有除法都使用安全除法，分母为 0 时结果设为 0。

In [ ]:
# 定义安全除法函数，避免分母为 0 时报错
def safe_divide(numerator, denominator):
    """如果分母为 0，则返回 0；否则返回 numerator / denominator。"""
    return numerator / denominator if denominator else 0


# 整理每个类目的基础指标
category_rows = []
for category_id in sorted(category_total_counter.keys()):
    pv_count = int(category_behavior_counter[category_id]["pv"])
    fav_count = int(category_behavior_counter[category_id]["fav"])
    cart_count = int(category_behavior_counter[category_id]["cart"])
    buy_count = int(category_behavior_counter[category_id]["buy"])

    pv_user_count = int(len(category_behavior_user_sets[category_id]["pv"]))
    fav_user_count = int(len(category_behavior_user_sets[category_id]["fav"]))
    cart_user_count = int(len(category_behavior_user_sets[category_id]["cart"]))
    buy_user_count = int(len(category_behavior_user_sets[category_id]["buy"]))

    category_rows.append(
        {
            "category_id": category_id,
            "total_behavior_count": int(category_total_counter[category_id]),
            "pv_count": pv_count,
            "fav_count": fav_count,
            "cart_count": cart_count,
            "buy_count": buy_count,
            "pv_user_count": pv_user_count,
            "fav_user_count": fav_user_count,
            "cart_user_count": cart_user_count,
            "buy_user_count": buy_user_count,
            "unique_item_count": int(len(category_item_sets[category_id])),
        }
    )

# 生成类目指标表
category_metrics = pd.DataFrame(category_rows)

# 新增类目转化指标
category_metrics["behavior_buy_rate"] = category_metrics.apply(lambda x: safe_divide(x["buy_count"], x["pv_count"]), axis=1)
category_metrics["user_buy_rate"] = category_metrics.apply(lambda x: safe_divide(x["buy_user_count"], x["pv_user_count"]), axis=1)
category_metrics["cart_user_rate"] = category_metrics.apply(lambda x: safe_divide(x["cart_user_count"], x["pv_user_count"]), axis=1)
category_metrics["fav_user_rate"] = category_metrics.apply(lambda x: safe_divide(x["fav_user_count"], x["pv_user_count"]), axis=1)
category_metrics["cart_to_buy_user_rate"] = category_metrics.apply(lambda x: safe_divide(x["buy_user_count"], x["cart_user_count"]), axis=1)
category_metrics["fav_to_buy_user_rate"] = category_metrics.apply(lambda x: safe_divide(x["buy_user_count"], x["fav_user_count"]), axis=1)

# 展示类目指标表
display(category_metrics.head())
print(f"类目指标表行数：{len(category_metrics):,}")

## 七、识别类目机会类型

这里改用“多榜单机会识别法”，不再把每个 `category_id` 强行划分为唯一机会类型。

真实业务中，一个类目可能同时存在多个机会点。例如某个类目既有较高加购量，又存在较低购买转化。多榜单法允许同一个类目出现在多个机会榜单中，更适合发现可运营机会。


In [ ]:
# 计算用于机会识别的分位数阈值
pv_75 = category_metrics["pv_count"].quantile(0.75)
pv_50 = category_metrics["pv_count"].quantile(0.50)
cart_75 = category_metrics["cart_count"].quantile(0.75)
fav_75 = category_metrics["fav_count"].quantile(0.75)
user_buy_rate_25 = category_metrics["user_buy_rate"].quantile(0.25)
user_buy_rate_75 = category_metrics["user_buy_rate"].quantile(0.75)
cart_to_buy_user_rate_25 = category_metrics["cart_to_buy_user_rate"].quantile(0.25)
fav_to_buy_user_rate_25 = category_metrics["fav_to_buy_user_rate"].quantile(0.25)

# 将阈值整理成字典，方便调试输出
thresholds = {
    "pv_75": pv_75,
    "cart_75": cart_75,
    "fav_75": fav_75,
    "user_buy_rate_25": user_buy_rate_25,
    "user_buy_rate_75": user_buy_rate_75,
    "cart_to_buy_user_rate_25": cart_to_buy_user_rate_25,
    "fav_to_buy_user_rate_25": fav_to_buy_user_rate_25,
}

# 定义机会长表需要保留的字段
opportunity_columns = [
    "category_id",
    "opportunity_type",
    "pv_count",
    "fav_count",
    "cart_count",
    "buy_count",
    "pv_user_count",
    "cart_user_count",
    "fav_user_count",
    "buy_user_count",
    "user_buy_rate",
    "cart_to_buy_user_rate",
    "fav_to_buy_user_rate",
]

# 1. 高浏览低购买类目榜单：浏览量高，但用户购买转化率低
high_pv_low_buy_categories = (
    category_metrics[
        (category_metrics["pv_count"] >= pv_75)
        & (category_metrics["user_buy_rate"] <= user_buy_rate_25)
        & (category_metrics["pv_user_count"] > 0)
    ]
    .sort_values("pv_count", ascending=False)
    .head(50)
    .copy()
)
high_pv_low_buy_categories["opportunity_type"] = "高浏览低购买类目"

# 2. 高加购低购买类目榜单：加购量高，但加购用户到购买用户转化偏低
high_cart_low_buy_categories = (
    category_metrics[
        (category_metrics["cart_count"] >= cart_75)
        & (category_metrics["cart_to_buy_user_rate"] <= cart_to_buy_user_rate_25)
        & (category_metrics["cart_user_count"] > 0)
    ]
    .sort_values("cart_count", ascending=False)
    .head(50)
    .copy()
)
high_cart_low_buy_categories["opportunity_type"] = "高加购低购买类目"

# 3. 高收藏低购买类目榜单：收藏量高，但收藏用户到购买用户转化偏低
high_fav_low_buy_categories = (
    category_metrics[
        (category_metrics["fav_count"] >= fav_75)
        & (category_metrics["fav_to_buy_user_rate"] <= fav_to_buy_user_rate_25)
        & (category_metrics["fav_user_count"] > 0)
    ]
    .sort_values("fav_count", ascending=False)
    .head(50)
    .copy()
)
high_fav_low_buy_categories["opportunity_type"] = "高收藏低购买类目"

# 4. 高转化优势类目榜单：浏览量不低，且用户购买转化率高
high_conversion_categories = (
    category_metrics[
        (category_metrics["pv_count"] >= pv_50)
        & (category_metrics["user_buy_rate"] >= user_buy_rate_75)
        & (category_metrics["buy_user_count"] > 0)
    ]
    .sort_values("user_buy_rate", ascending=False)
    .head(50)
    .copy()
)
high_conversion_categories["opportunity_type"] = "高转化优势类目"

# 检查是否有空榜单；如果为空，打印阈值帮助调试
opportunity_lists = {
    "高浏览低购买类目": high_pv_low_buy_categories,
    "高加购低购买类目": high_cart_low_buy_categories,
    "高收藏低购买类目": high_fav_low_buy_categories,
    "高转化优势类目": high_conversion_categories,
}

for opportunity_type, df in opportunity_lists.items():
    if df.empty:
        print(f"【调试提示】{opportunity_type} 榜单为空，当前筛选阈值如下：")
        for name, value in thresholds.items():
            print(f"{name}: {value}")
        print("说明：如果 DEBUG=True，只处理前几个 chunk，样本不足可能导致部分榜单为空；可先检查阈值，确认逻辑后再跑全量数据。\n")

# 合并四张机会榜单为长表；同一个 category_id 可以出现在多个 opportunity_type 中，不去重
category_opportunity = pd.concat(
    [
        high_pv_low_buy_categories[opportunity_columns],
        high_cart_low_buy_categories[opportunity_columns],
        high_fav_low_buy_categories[opportunity_columns],
        high_conversion_categories[opportunity_columns],
    ],
    ignore_index=True,
)

# 统计四类机会榜单数量
opportunity_counts = (
    category_opportunity["opportunity_type"]
    .value_counts()
    .reindex(opportunity_order)
    .fillna(0)
    .astype(int)
)

# 展示机会榜单数量和机会长表
print("四类机会榜单数量：")
display(opportunity_counts)
display(category_opportunity.head())

# 打印关键阈值，便于理解规则边界
print("机会识别阈值：")
for name, value in thresholds.items():
    print(f"{name}: {value:.6f}")


## 八、生成类目 × 用户分群行为表

将分块累计的 `category_id × user_segment × behavior_type` 行为数量整理成宽表，并为后续策略表找出每个类目的主要流量来源用户群体和主要购买贡献用户群体。

In [ ]:
# 整理类目 × 用户分群行为表
segment_rows = []
for (category_id, user_segment), counter in category_segment_behavior_counter.items():
    pv_count = int(counter["pv"])
    fav_count = int(counter["fav"])
    cart_count = int(counter["cart"])
    buy_count = int(counter["buy"])

    segment_rows.append(
        {
            "category_id": category_id,
            "user_segment": user_segment,
            "pv_count": pv_count,
            "fav_count": fav_count,
            "cart_count": cart_count,
            "buy_count": buy_count,
            "total_behavior_count": pv_count + fav_count + cart_count + buy_count,
        }
    )

# 生成类目 × 用户分群行为 DataFrame
category_segment_behavior = pd.DataFrame(segment_rows)

# 如果没有匹配到用户分群，给出明确提示
if category_segment_behavior.empty:
    raise ValueError("没有匹配到任何用户分群行为，请检查 user_segmentation.csv 的 user_id 是否与原始数据一致。")

# 找出每个类目的主要流量来源用户群体
main_pv_segment = (
    category_segment_behavior.sort_values(["category_id", "pv_count"], ascending=[True, False])
    .drop_duplicates(subset=["category_id"])
    [["category_id", "user_segment"]]
    .rename(columns={"user_segment": "main_pv_user_segment"})
)

# 找出每个类目的主要购买贡献用户群体
main_buy_segment = (
    category_segment_behavior.sort_values(["category_id", "buy_count"], ascending=[True, False])
    .drop_duplicates(subset=["category_id"])
    [["category_id", "user_segment"]]
    .rename(columns={"user_segment": "main_buy_user_segment"})
)

# 展示类目 × 用户分群行为表
display(category_segment_behavior.head())
print(f"类目 × 用户分群行为表行数：{len(category_segment_behavior):,}")

## 九、生成类目策略建议表

将类目机会类型和用户分群行为结合，输出每个类目的主要问题、主要流量用户群体、主要购买用户群体和对应运营策略。

In [ ]:
# 为不同机会类型配置业务问题和策略建议
strategy_map = {
    "高浏览低购买类目": {
        "business_problem": "类目浏览量较高，但购买转化率偏低，说明用户有兴趣但下单意愿不足，可能存在价格、详情页信息、评价内容或推荐精准度问题。",
        "strategy_recommendation": "重点分析高浏览低转化用户，优化商品详情页、价格展示、评价内容和推荐匹配度，可通过优惠券和相似商品推荐提升转化。",
    },
    "高加购低购买类目": {
        "business_problem": "用户已经产生较强购买意向，但最终购买不足，可能存在价格敏感、库存、运费、优惠不足等问题。",
        "strategy_recommendation": "针对高潜力用户和加购未买用户推送限时优惠券、降价提醒、库存提醒和满减活动，促进临门一脚转化。",
    },
    "高收藏低购买类目": {
        "business_problem": "用户对商品有兴趣但决策周期较长，可能仍在比较或观望。",
        "strategy_recommendation": "针对收藏未买用户推送买家秀、评价内容、相似商品对比和收藏商品降价提醒，降低决策成本。",
    },
    "高转化优势类目": {
        "business_problem": "类目转化表现较好，说明商品匹配度和购买意愿较强。",
        "strategy_recommendation": "可向高价值用户和普通购买用户做复购推荐、搭配推荐和会员专属活动，提高复购和客单贡献。",
    },
}

# 合并主要流量来源和主要购买贡献用户群体
category_strategy_recommendations = (
    category_opportunity
    .merge(main_pv_segment, on="category_id", how="left")
    .merge(main_buy_segment, on="category_id", how="left")
)

# 增加业务问题和策略建议字段
category_strategy_recommendations["business_problem"] = category_strategy_recommendations["opportunity_type"].map(lambda x: strategy_map[x]["business_problem"])
category_strategy_recommendations["strategy_recommendation"] = category_strategy_recommendations["opportunity_type"].map(lambda x: strategy_map[x]["strategy_recommendation"])

# 保留题目要求的字段
strategy_columns = [
    "category_id",
    "opportunity_type",
    "pv_count",
    "cart_count",
    "fav_count",
    "buy_count",
    "user_buy_rate",
    "main_pv_user_segment",
    "main_buy_user_segment",
    "business_problem",
    "strategy_recommendation",
]
category_strategy_recommendations = category_strategy_recommendations[strategy_columns]

# 展示策略建议表
display(category_strategy_recommendations.head())


## 十、保存结果

保存四张结果表到 `E:\ecommerce-user-growth-analysis\data\processed\`。

In [ ]:
# 保存类目指标表
category_metrics.to_csv(category_metrics_path, index=False, encoding="utf-8-sig")

# 保存四张独立机会榜单
high_pv_low_buy_categories.to_csv(high_pv_low_buy_path, index=False, encoding="utf-8-sig")
high_cart_low_buy_categories.to_csv(high_cart_low_buy_path, index=False, encoding="utf-8-sig")
high_fav_low_buy_categories.to_csv(high_fav_low_buy_path, index=False, encoding="utf-8-sig")
high_conversion_categories.to_csv(high_conversion_path, index=False, encoding="utf-8-sig")

# 保存类目机会长表，同一个 category_id 可以出现在多个 opportunity_type 中
category_opportunity.to_csv(category_opportunity_path, index=False, encoding="utf-8-sig")

# 保存类目 × 用户分群行为表
category_segment_behavior.to_csv(category_segment_behavior_path, index=False, encoding="utf-8-sig")

# 保存类目策略建议表
category_strategy_recommendations.to_csv(strategy_path, index=False, encoding="utf-8-sig")

print("结果文件已保存：")
print(category_metrics_path)
print(category_opportunity_path)
print(high_pv_low_buy_path)
print(high_cart_low_buy_path)
print(high_fav_low_buy_path)
print(high_conversion_path)
print(category_segment_behavior_path)
print(strategy_path)


## 十一、可视化

下面围绕四张机会榜单重新绘图，重点观察各机会类型的榜单数量，以及每类机会 Top 20 类目的排序。


In [ ]:
# 1. 四类机会榜单数量柱状图
opportunity_count_df = opportunity_counts.reset_index()
opportunity_count_df.columns = ["opportunity_type", "category_count"]

plt.figure(figsize=(10, 5))
sns.barplot(data=opportunity_count_df, x="opportunity_type", y="category_count", color="#5F8D4E")
plt.title("四类机会榜单数量")
plt.xlabel("机会类型")
plt.ylabel("类目数量")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 2. 高浏览低购买 Top 20 类目柱状图
top20_high_pv_low_buy = high_pv_low_buy_categories.sort_values("pv_count", ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20_high_pv_low_buy, x="category_id", y="pv_count", color="#2F6B9A")
plt.title("高浏览低购买 Top 20 类目")
plt.xlabel("类目 ID")
plt.ylabel("浏览量")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 3. 高加购低购买 Top 20 类目柱状图
top20_high_cart_low_buy = high_cart_low_buy_categories.sort_values("cart_count", ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20_high_cart_low_buy, x="category_id", y="cart_count", color="#D08C60")
plt.title("高加购低购买 Top 20 类目")
plt.xlabel("类目 ID")
plt.ylabel("加购量")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 4. 高收藏低购买 Top 20 类目柱状图
top20_high_fav_low_buy = high_fav_low_buy_categories.sort_values("fav_count", ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20_high_fav_low_buy, x="category_id", y="fav_count", color="#7B6D8D")
plt.title("高收藏低购买 Top 20 类目")
plt.xlabel("类目 ID")
plt.ylabel("收藏量")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 5. 高转化优势 Top 20 类目柱状图
top20_high_conversion = high_conversion_categories.sort_values("user_buy_rate", ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20_high_conversion, x="category_id", y="user_buy_rate", color="#B85C38")
plt.title("高转化优势 Top 20 类目")
plt.xlabel("类目 ID")
plt.ylabel("用户购买转化率")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 6. 不同用户分群在重点机会类目中的行为贡献图
# 重点机会类目来自四张机会榜单，取浏览量最高的前 20 个类目，避免图表过于拥挤
focus_categories = (
    category_opportunity.sort_values("pv_count", ascending=False)
    .head(20)["category_id"]
    .drop_duplicates()
)

focus_segment_behavior = category_segment_behavior[category_segment_behavior["category_id"].isin(focus_categories)]

# 按用户分群汇总重点机会类目中的四类行为贡献
focus_segment_summary = (
    focus_segment_behavior
    .groupby("user_segment")[["pv_count", "fav_count", "cart_count", "buy_count"]]
    .sum()
    .reset_index()
)

focus_segment_melt = focus_segment_summary.melt(
    id_vars="user_segment",
    value_vars=["pv_count", "fav_count", "cart_count", "buy_count"],
    var_name="行为类型",
    value_name="行为数量",
)

plt.figure(figsize=(13, 6))
sns.barplot(data=focus_segment_melt, x="user_segment", y="行为数量", hue="行为类型")
plt.title("不同用户分群在重点机会类目中的行为贡献")
plt.xlabel("用户分群")
plt.ylabel("行为数量")
plt.xticks(rotation=30, ha="right")
plt.legend(title="行为类型")
plt.tight_layout()
plt.show()


## 十二、Markdown 业务解读

下面基于多榜单机会识别结果和用户分群联动结果，自动生成业务解读。


In [ ]:
# 提取各类机会的 Top 类目 ID，注意只展示数字编号，不编造真实类目名称
def top_category_text(df, sort_col, top_n=5):
    """返回某张机会榜单下 Top N 类目 ID 文本。"""
    top_categories = (
        df.sort_values(sort_col, ascending=False)
        .head(top_n)["category_id"]
        .astype(str)
        .tolist()
    )
    return "、".join(top_categories) if top_categories else "暂无"


high_pv_low_buy_text = top_category_text(high_pv_low_buy_categories, "pv_count")
high_cart_low_buy_text = top_category_text(high_cart_low_buy_categories, "cart_count")
high_fav_low_buy_text = top_category_text(high_fav_low_buy_categories, "fav_count")
high_conversion_text = top_category_text(high_conversion_categories, "user_buy_rate")

# 找出重点机会类目中行为贡献最高的用户分群
if not focus_segment_summary.empty:
    top_pv_segment = focus_segment_summary.sort_values("pv_count", ascending=False).iloc[0]["user_segment"]
    top_cart_segment = focus_segment_summary.sort_values("cart_count", ascending=False).iloc[0]["user_segment"]
    top_buy_segment = focus_segment_summary.sort_values("buy_count", ascending=False).iloc[0]["user_segment"]
else:
    top_pv_segment = "暂无"
    top_cart_segment = "暂无"
    top_buy_segment = "暂无"

# 使用 Markdown 输出业务解读
display(
    Markdown(
        f"""
### 1. 为什么要做商品类目机会分析？

商品类目机会分析可以把平台增长问题从整体流量拆解到具体类目。不同类目可能分别存在高流量低转化、高加购未购买、高收藏观望或高转化优势等问题，拆到类目后才能制定更具体的推荐、优惠和内容优化策略。

### 2. 为什么采用多榜单识别法？

本部分采用 **多榜单识别法**，而不是互斥分类法。原因是：真实业务中，一个类目可能同时存在多个机会点，例如既有高加购，又有低购买转化。多榜单法允许同一个 `category_id` 同时出现在多个机会榜单中，更适合发现可运营机会。

### 3. 为什么类目分析要和用户分群结合？

单看类目只能知道哪里有机会，结合用户分群才能知道机会来自哪类用户。例如同样是高浏览低购买类目，如果主要流量来自高浏览低转化用户，重点应优化推荐和详情页；如果主要购买来自高价值用户，则可以做复购和会员权益承接。

### 4. 哪些类目属于高浏览低购买？

按当前规则识别出的高浏览低购买类目 Top 示例为：**{high_pv_low_buy_text}**。这些类目有较高浏览量，但用户购买转化偏低，需要重点排查价格、详情页、评价内容和推荐匹配问题。

### 5. 哪些类目属于高加购低购买？

高加购低购买类目 Top 示例为：**{high_cart_low_buy_text}**。这些类目说明用户已经有较强购买意向，但最后成交不足，适合使用限时优惠、降价提醒、库存提醒和满减活动推动转化。

### 6. 哪些类目属于高收藏低购买？

高收藏低购买类目 Top 示例为：**{high_fav_low_buy_text}**。这类用户往往处在比较或观望阶段，可以通过买家秀、评价内容、相似商品对比和收藏商品降价提醒降低决策成本。

### 7. 哪些类目属于高转化优势类目？

高转化优势类目 Top 示例为：**{high_conversion_text}**。这些类目的用户购买意愿更强，可重点用于复购推荐、搭配推荐和会员专属活动，提高复购和客单贡献。

### 8. 不同用户分群在类目机会中扮演什么角色？

在重点机会类目中，浏览贡献较高的用户分群是 **{top_pv_segment}**，加购贡献较高的用户分群是 **{top_cart_segment}**，购买贡献较高的用户分群是 **{top_buy_segment}**。这说明类目增长策略应同时区分流量来源、购买意向和最终成交贡献。

### 9. 平台应该如何针对不同类目和用户群体制定增长策略？

高浏览低购买类目应优先优化详情页、价格展示和推荐匹配；高加购低购买类目应重点使用优惠券、降价提醒和购物车召回；高收藏低购买类目应强化内容种草、评价和相似商品对比；高转化优势类目应加大对高价值用户和普通购买用户的复购推荐。所有策略都应结合主要流量来源用户群体和主要购买贡献用户群体制定。
        """
    )
)
